# Experiment description

Here, we'll calculate viral titer (transducing units per volume) for several bioreps (i.e., two separate batches) of several different lentiviral vectors.

These experiments followed the [measuring viral titer protocol](https://gallowaylabmit.github.io/protocols/en/latest/protocols/tc/virus/viral_titer.html), in which a known number of cells are transduced with a serial dilution (known volumes) of each virus. By measuring fluorescence of each cargo—these are two-gene vectors expressing both mRuby2 and mGL—we can find the fraction of expressing (transduced) cells in each condition. Then, we can fit a curve to this data (independent variable: volume of virus, dependent variable: fraction of cells transduced) assuming a Poisson process for transduction. The result is a measure of viral titer, in transducing units per volume, for each batch of each vector.

Note that this is a *functional* metric for viral titer; it is likely correlated with the concentration of correctly formed viral particles, but it is not equivalent. This metric also takes into account infection, integration, and expression steps.

Finally, included at the end is a calculation of the virus volume needed for the next experiment, in which we want to transduce cells at a particular multiplicity of infection (MOI). This uses the computed viral titer and desired experimental parameters to find the volume of virus to add to each condition.

# Load data

We'll load data from two titer experiments. The relevant metadata are:

- `construct`: the transfer plasmid used in virus production (i.e., what is contained in the viral vector)
    
    - You could also call this `vector` or `plasmid`
    - We also included an untransduced condition (`UT`) to set gates

- `scaling`: the relative dilution of the virus, to be combined with a `max_virus` value to specify the volume of virus added to each well

    -  It is also possible to directly specify volumes of virus added to each condition, rather than separate `scaling` and `max_virus` parameters. 

- `replicate`: technical replicate (each virus condition was mixed in one vessel but pipetted into two wells of cells)
- `biorep`: separate batch of each vector|
- `max_virus`: the volume of virus added to the first dilution; this was the same for all vectors
- `cell_count`: the number of cells at the time of transduction; this was the same for all vectors

    - The number of cells at the time of transduction is known directly if infecting in suspension
    - If transducing cells previously plated, you can either (1) estimate the number of cells by assuming a doubling time of 24 hours (e.g., 2x the seeding amount if cells were transduced 24 hours later) or (2) dissociate and count several wells

In [ ]:
# Import our favorite packages
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rushd as rd
import scipy as sp
import seaborn as sns

# Set plotting style
sns.set_style('ticks')
sns.set_context('talk',rc={'font.family': 'sans-serif', 'font.sans-serif':['Helvetica Neue']})

In [ ]:
# Set up data loading and check .yaml files
base_path = rd.datadir/'instruments'/'data'/'attune'/'kasey'
plates = pd.DataFrame({
    'data_path': [base_path/'2025.10.25_exp146'/'export'/f'plate{n+1}' for n in range(2)] + 
                 [base_path/'2025.12.02_exp146.0'/'export'],
    'yaml_path': [base_path/'2025.10.25_exp146'/'export'/f'wells_plate{n+1}.yaml' for n in range(2)] + 
                 [base_path/'2025.12.02_exp146.0'/'export'/'wells.yaml'],
    'max_virus': [4]*3, # in uL
    'cell_count': [2e4]*3,
})

for p in plates['yaml_path'].unique():
    rd.plot.plot_well_metadata(p)

In [ ]:
# Load data
output_path = rd.rootdir/'output'/'flow-example_viral-titer'
cache_path = output_path/'data.gzip'
channel_list = ['mRuby2-A','mGL-A']
data = pd.DataFrame()

if cache_path.exists(): data = pd.read_parquet(cache_path)
else: 
    data = rd.flow.load_groups_with_metadata(plates, columns=channel_list)
    data.to_parquet(rd.outfile(cache_path))
display(data)

# Gate cells and visualize distributions

In [ ]:
# Draw gates using untransduced condition
gates = data[data['construct']=='UT'].groupby(['biorep'])[channel_list].apply(lambda x: x.quantile(0.995)).reset_index()
display(gates)

In [ ]:
# Visualize distributions 
x = 'mGL-A'
y = 'mRuby2-A'
for biorep, group in data.groupby('biorep'):

    # Just plot the max virus amount,
    # only showing positive (>0) channel values since plotting on log scale
    # and subsampling so it plots faster
    plot_df = group[(group['scaling']==1) & (group[x]>0) & (group[y]>0)].groupby('construct').sample(1000)

    g = sns.displot(data=plot_df, x=x, y=y, col='construct', col_wrap=4, kind='kde',
                    log_scale=True, common_norm=False, fill=False, levels=7,
                    hue='construct')
    
    # Add references lines for gates
    gate = gates[(gates['biorep']==biorep)]
    if gate.empty: continue

    for _, ax in g.axes_dict.items():
        ax.axvline(gate[x].values[0], color='black', zorder=0)
        ax.axhline(gate[y].values[0], color='black', zorder=0)

    g.figure.savefig(rd.outfile(output_path/f'kde_mGL-mRuby2_biorep{biorep}.png'))

From these plots, we notice that pLEN145 has very few expressing cells, while pLEN233 transduced all cells with the maximum virus amount.

In [ ]:
# Visualize distributions in one channel, plotting all virus amounts
x = 'mRuby2-A'
for biorep, group in data.groupby('biorep'):

    # Only show positive (>0) channel values since plotting on log scale
    # and subsample so it plots faster
    plot_df = group[group[x]>0].groupby('construct').sample(1000)

    g = sns.displot(data=plot_df, x=x, col='construct', col_wrap=4, kind='kde',
                    log_scale=True, common_norm=False, 
                    hue='scaling', palette='viridis', hue_norm=matplotlib.colors.FuncNorm((np.log, np.exp)))
    
    # Add references lines for gates
    gate = gates[(gates['biorep']==biorep)]
    if gate.empty: continue

    for _, ax in g.axes_dict.items():
        ax.axvline(gate[x].values[0], color='black', zorder=0)

    g.figure.savefig(rd.outfile(output_path/f'kde_mRuby2_biorep{biorep}.png'))

In these plots, we mostly see distributions shifting unimodally as the amount of virus changes. This is not quite expected: for vectors with clear expressing populations (e.g., high levels of the FP even at single copy), we'd expect to observe a single expressing peak (in addition to the not expressing peak, i.e., a bimodal distribution), where the fraction of cells in the expressing peak changes with virus amount. In fact, this is what we see for the mGL distributions below, just not for the mRuby2 ones above. This indicates that we're unable to distinguish expressing from not expressing cells well in the mRuby2 channel—here likely because mRuby2 levels are so low. (Indeed, these vectors mostly have the hPGK driving mRuby2 expression, which we know is weak in HEK293T cells.) This is one reason why it's convenient to categorize cells as transduced if they express either gene, rather than gating on one channel.

For vectors with very high titer, we would expect to observe a unimodal distribution at high virus amounts, as the entire population of cells may be transduced, just with varying copy number (and thus different expression levels). In fact, we do see this for pLEN233 and pLEN235 in the mGL distributions below. As the virus amount decreases, the distribution becomes bimodal with a non-expressing population, as expected.

In [ ]:
# Visualize distributions in one channel, plotting all virus amounts
x = 'mGL-A'
for biorep, group in data.groupby('biorep'):

    # Only show positive (>0) channel values since plotting on log scale
    # and subsample so it plots faster
    plot_df = group[group[x]>0].groupby('construct').sample(1000)

    g = sns.displot(data=plot_df, x=x, col='construct', col_wrap=4, kind='kde',
                    log_scale=True, common_norm=False, 
                    hue='scaling', palette='viridis', hue_norm=matplotlib.colors.FuncNorm((np.log, np.exp)))
    
    # Add references lines for gates
    gate = gates[(gates['biorep']==biorep)]
    if gate.empty: continue

    for _, ax in g.axes_dict.items():
        ax.axvline(gate[x].values[0], color='black', zorder=0)

    g.figure.savefig(rd.outfile(output_path/f'kde_mRuby2_biorep{biorep}.png'))

To compute titer, we'll count a cell as transduced if it expresses either (or both) of the two genes.

In [ ]:
# Categorize values into quadrants (0,1,2,3) using gates for two channels
def get_quadrant(df, cols, gates, by):
    '''
    Possible values:
        0 = double negative
        1 = x-positive
        2 = y-positive
        3 = double positive
    '''
    this = df[by].values[0]
    df['x'] = df[cols[0]] > gates.loc[(gates[by]==this), cols[0]].values[0]
    df['y'] = df[cols[1]] > gates.loc[(gates[by]==this), cols[1]].values[0]
    df['quadrant'] = df['x'].astype(int) + df['y'].astype(int)*2
    return df

data = data.groupby('biorep')[data.columns].apply(get_quadrant, channel_list, gates, 'biorep').reset_index(drop=True)

# Compute fraction expressing (not double negative, i.e., expressing either FP)
by = ['biorep','construct','scaling','replicate','max_virus','cell_count']
fraction = (data[data.quadrant>0].groupby(by)['mGL-A'].count() / 
            data.groupby(by)['mGL-A'].count()).reset_index().rename(columns={'mGL-A': 'fraction_transduced'}).dropna()
display(fraction)

# Calculate titer

In [ ]:
# Combine parameters to compute virus amount in each condition
fraction['virus_uL'] = fraction['max_virus'] * fraction['scaling']

# Combine technical replicates in a new dataframe
# (we want to curve fit on one point per condition)
by = ['biorep','construct','scaling','max_virus','virus_uL','cell_count']
fraction_avg = fraction.groupby(by)[['fraction_transduced']].mean().reset_index()

Optionally, we can remove points that look out of place—namely, conditions with lower fraction expressing for higher virus volume. This could be due to technical error (or possibly some kind of underlying biology that we don't understand).

In [ ]:
# Add a column to denote whether values in 'decreasing_col' decrease with 'sort_col'
# e.g., label conditions (construct + virus vol) as decreasing if any fraction values
#   decrease as virus volume increases
def find_decreasing(df, sort_col, decreasing_col):
    sorted = list(df.sort_values(sort_col)[decreasing_col])
    df['decreasing'] = False
    cur_max = sorted[0]
    for i in range(len(sorted)):
        if sorted[i] < cur_max:
            df.loc[df[decreasing_col]==sorted[i], 'decreasing'] = True
        else:
            cur_max = sorted[i]
    return df

# Label decreasing points
by = ['biorep','construct','cell_count']
fraction_avg = fraction_avg.groupby(by).apply(find_decreasing, 'virus_uL', 'fraction_transduced').reset_index().drop(columns=f'level_{len(by)}')
display(fraction_avg)

Fit virus volume and fraction transduced to a Poisson distribution. For a derivation of this formula, see the description in the [measuring viral titer protocol](https://gallowaylabmit.github.io/protocols/en/latest/protocols/tc/virus/viral_titer.html#computing-titer-from-transduction-data).

In [ ]:
# Models P(transduced) assuming a Poisson distribution
def poisson_model(virus_vol, tu_per_cell_per_vol):
    return 1 - np.exp(-tu_per_cell_per_vol * virus_vol)

# Perform curve fit
def fit_poisson(df, x_col, y_col):
    popt, pcov, infodict, _, _ = sp.optimize.curve_fit(poisson_model, df[x_col], df[y_col], p0=0.5, bounds=(0, np.inf), full_output=True)
    return pd.DataFrame({'tu_per_cell_per_uL': [popt[0]], 'perr': [np.sqrt(np.diag(pcov))[0]], 'mse': np.square(infodict['fvec']).mean()})


# Fit fraction transduced to Poisson distribution
# ignore conditions labeled as 'decreasing' above, unless they are from the untransduced condition
fits = fraction_avg[~(fraction_avg.decreasing) | (fraction_avg.construct=='UT')].groupby(by).apply(fit_poisson, 'virus_uL', 'fraction_transduced', include_groups=False).reset_index().drop(columns=f'level_{len(by)}')

# Convert fitted TU/cell/uL to titer (TU/uL) using known cell counts
fits['tu_per_uL'] = fits['tu_per_cell_per_uL'] * fits['cell_count']

display(fits)

In [ ]:
# Verify titer fits by plotting data + curve
plot_df = fraction
x_fit = np.linspace(0,100,10000)
titer_biorep_palette = {biorep: color for biorep, color in zip(fits.biorep.unique(), sns.color_palette("husl", len(fits.biorep.unique())))}
g = sns.relplot(data=plot_df, x='virus_uL', y='fraction_transduced', col='construct', col_wrap=4, kind='scatter', 
                legend=False, hue='biorep', palette=titer_biorep_palette, facet_kws={'margin_titles': True},
                height=5, aspect=0.9)

for construct, ax in g.axes_dict.items():
    for biorep, df in fits[(fits.construct==construct)].groupby('biorep'):
        y_fit = poisson_model(x_fit, df['tu_per_cell_per_uL'].mean())
        ax.plot(x_fit, y_fit, color=titer_biorep_palette[biorep], label='fit', zorder=0)
    ax.set(xscale='log', xlabel='Virus volume (µL/well)', ylabel='Fraction transduced')

g.figure.savefig(rd.outfile(output_path/f'titer_fits.png'), bbox_inches='tight')

Looking at the plots, we see that the untransduced condition has a curve fit that increases at high virus volumes (much higher than tested here). While of course we would expect titer measurements of zero for this condition, in reality we will not observe this: there will, by definition, be cells counted as transduced because we set gates based on the 99.5th percentile (namely, ~0.5% of cells). Then, we force the curve fit to make an estimate of the titer from this data—no doubt a poor fit. However, this value is useful in one sense: it sets the floor of titer values, i.e., if calculated titers are close to those (nonsense ones) for the untransduced condition, then the titer is effectively zero.

We see something like this for pLEN145: there may be a small population of actually transduced cells, but the titer value is quite low. This is even more apparent in the next plot, where we compare the computed titer values across vectors.

Otherwise, the curve fits look pretty good, and it doesn't seem like we needed to throw out any outlier points.

Next, we can plot the titers computed for each vector to facilitate comparison. If you don't care about the titer values themselves but just need them for subsequent experiments, you don't need to make these plots. In that case, you could skip to the next section to calculate volumes to pipet.

In [ ]:
# Compare titers
f, ax = plt.subplots(1,1, figsize=(4,4), gridspec_kw=dict(wspace=0.4),)
plot_df = fits
y = 'tu_per_uL'

sns.stripplot(data=plot_df, x='construct', y=y, ax=ax, hue='biorep', palette=titer_biorep_palette,
              legend=False, size=7)
sns.despine(ax=ax)
ax.axhline(fits.loc[fits.construct=='UT', y].mean(), c='k', ls=':')
ax.set(xlabel='', ylabel='Titer (TU/uL)', yscale='log')
ax.set_xticklabels(ax.get_xticklabels(), rotation=90)

f.savefig(rd.outfile(output_path/f'titer_points.png'), bbox_inches='tight')

In [ ]:
# Compare titers
f, ax = plt.subplots(1,1, figsize=(4,4), gridspec_kw=dict(wspace=0.4),)
plot_df = fits
y = 'tu_per_uL'

sns.pointplot(data=plot_df, x='construct', y=y, ax=ax,
              legend=False, estimator='mean', errorbar='se', linestyle='none',
              capsize=0.1, ms=8, mec='white', mew=1, err_kws={'linewidth': 2, 'zorder': 0})
sns.despine(ax=ax)
ax.axhline(fits.loc[fits.construct=='UT', y].mean(), c='k', ls=':', zorder=0)
ax.set(xlabel='', ylabel='Titer (TU/uL)', yscale='log')
ax.set_xticklabels(ax.get_xticklabels(), rotation=90)

f.savefig(rd.outfile(output_path/f'titer_errorbars.png'), bbox_inches='tight')

Are these values different from each other? By how much, exactly? When comparing vectors, we are more interested in relative titers than absolute values—if you care about titer at all, that is.

In [ ]:
# Helper functions for annotating stats
star_convert = list(reversed([(1.0, 'n.s.'), (0.05, '*'), (0.01, '**'), (0.001, '***'), (0.0001, '****')]))
pval_to_stars = lambda x: next(val[1] for val in star_convert if val[0] > x)

In [ ]:
f, ax = plt.subplots(1,1, figsize=(4,4), gridspec_kw=dict(wspace=0.4),)
plot_df = fits
y = 'tu_per_uL'

sns.pointplot(data=plot_df, x='construct', y=y, ax=ax,
              legend=False, estimator='mean', errorbar='se', linestyle='none',
              capsize=0.1, ms=8, mec='white', mew=1, err_kws={'linewidth': 2, 'zorder': 0})
sns.despine(ax=ax)
ax.axhline(fits.loc[fits.construct=='UT', y].mean(), c='k', ls=':', zorder=0)
ax.set(xlabel='', ylabel='Titer (TU/uL)', yscale='log')
ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
order = [l.get_text() for l in ax.get_xticklabels()]

# T-tests, comparing EFS only
pairs = [('pLEN232','pLEN233'), ('pLEN234','pLEN235'), ('pLEN233','pLEN235')]
yloc_top = 0.99
for pair, offset in zip(pairs,[0,0.1,0.1]):
    val1 = plot_df[(plot_df.construct==pair[0])][y]
    val2 = plot_df[(plot_df.construct==pair[1])][y]
    xloc1 = order.index(pair[0])
    xloc2 = order.index(pair[1])
    pad = 0.02
    
    star = pval_to_stars(sp.stats.ttest_ind(val1, val2,).pvalue)
    print(f'{pair}: {star}')
    if star == 'n.s.': continue
    yloc = yloc_top - offset
    t = ax.text(xloc1+(xloc2-xloc1)/2, yloc - pad * (star.startswith('*')), star, 
                transform=ax.get_xaxis_transform(), zorder=0, ha='center', va='center', 
                backgroundcolor='white', bbox={"facecolor": 'white', "pad": 1})
    ax.add_artist(matplotlib.lines.Line2D([xloc1, xloc2], [yloc, yloc], zorder=-1,
                                          transform=ax.get_xaxis_transform(), color='k', linewidth=1))
    fc = val2.mean() / val1.mean()
    print(f'{fc:0.2f}')
    t = ax.text(np.mean([xloc1,xloc2]), yloc+0.04, f'{fc:0.1f}x', size='small',
                transform=ax.get_xaxis_transform(), ha='center', va='center', zorder=0)

# Calculate volumes to pipet in next experiments

Now that you know the viral titer, you can use this information to calculate the virus volumes to pipet in subsequent experiments. While you can do these calculations in Excel or elsewhere, sometimes it's nice to do them in Python, since the titer values are already loaded.

In particular, what you'll want to calculate is the volume of virus to add for the given parameters:

- `cell_count`: number of cells to transduce per well
- `moi`: multiplicity of infection, the ratio of transducing units to cells

    - Use MOI $\leq$ 0.3 to ensure single integrations
    - Use MOI = 3 (or greater) to transduce the vast majority of cells

- `tu_per_uL`: titer calculated above, unique for each batch of virus
- `num_wells`: number of wells per condition
- `extra`: multiplier to prepare extra solution to account for pipetting loss

In [ ]:
# Example calculation
cell_count = 2e4    # #/well
moi = 0.3           # TU/cell
num_wells = 3
extra = 1.1

result = fits[['biorep', 'construct', 'tu_per_uL']].copy()
result['virus_uL'] = num_wells * extra * cell_count * moi / result['tu_per_uL']
display(result)

Notice that the untransduced condition would require >200 uL of "virus" per well, which is more than was made for each vector. (The production protocol for these viruses used per batch: one 10cm dish per vector, resuspending concentrated virus in 200 uL.) And notice that pLEN145 biorep 3 also requires >200 uL, indicating that this vector failed to produce. This condition will be excluded from future experiments, as there is not enough virus; alternatively, if it is really important to include, you could scale down the number of wells to accomodate.

In [ ]:
# display table most convenient for copy-pasting / printing
display(result.loc[result.construct!='UT', ['biorep','construct','virus_uL']])